In [9]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [10]:
import torch

print("PyTorch version:", torch.__version__)


PyTorch version: 2.8.0.dev20250319+cu128


In [11]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import torchvision.models as models
import torchvision

from torchvision import transforms
import time
torch.manual_seed(17)

In [12]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
print(device)

cuda


In [14]:
transform = transforms.Compose([ 
    transforms.Resize((224, 224)), 
    transforms.ToTensor()
])

In [15]:
dataset = torchvision.datasets.ImageFolder('./tiny-imagenet-200/train', transform=transform)

In [16]:
len(dataset)

100000

In [17]:
#split the data
train_data, val_data, test_data = torch.utils.data.random_split(dataset, [80000, 10000, 10000])

In [18]:
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

In [19]:
# Load pretrained ResNet-50 (Teacher Model)
teacher = models.resnet50(pretrained=True)

# Modify the final fully connected layer for 10 classes (CIFAR-10)
teacher.fc = nn.Linear(teacher.fc.in_features, 200)
# Move models to device
teacher = teacher.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
24.3%

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100.0%


In [20]:

model_path = 'best_teacher_model.pth'
# Load the model weights
teacher.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

<All keys matched successfully>

In [21]:
# Load pretrained ResNet-18 (Student Model)
student = models.resnet18(pretrained=True)
# Modify the final fully connected layer for 10 classes (CIFAR-10)
student.fc = nn.Linear(student.fc.in_features, 200)
student = student.to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
65.5%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


In [18]:

# model_path = 'student_before_pruning.pth'
# # Load the model weights
# student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

<All keys matched successfully>

In [22]:
# Logits normalization function
def normalize(logit):
    mean = logit.mean(dim=-1, keepdim=True)
    stdv = logit.std(dim=-1, keepdim=True)
    return (logit - mean) / (1e-7 + stdv)


In [23]:
# CA-KLD Loss for Classification
def cakld_loss(student_logits, teacher_logits, beta_prob):
    # Forward KL (student || teacher)
    student_log_prob = F.log_softmax(student_logits, dim=1)
    teacher_prob = F.softmax(teacher_logits, dim=1)
    forward_kl = F.kl_div(student_log_prob, teacher_prob, reduction='batchmean')

    # Reverse KL (teacher || student)
    teacher_log_prob = F.log_softmax(teacher_logits, dim=1)
    student_prob = F.softmax(student_logits, dim=1)
    reverse_kl = F.kl_div(teacher_log_prob, student_prob, reduction='batchmean')

    # Combined KL loss
    kl_loss = beta_prob * reverse_kl + (1 - beta_prob) * forward_kl
    return kl_loss


In [24]:
def evaluate(model, test_loader, device):
    model = model.to(device)  # Ensure model is on the correct device
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [25]:
def calculate_sparsity(model):
    total_zeros = 0
    total_params = 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            total_zeros += torch.sum(param == 0).item()
            total_params += param.numel()
    return total_zeros / total_params

In [26]:
import torch
import time
def measure_inference_time(model, test_loader, num_runs=5):
    device = torch.device('cpu')
    model.eval()
    model.to(device)

    # Warm-up (one batch to avoid startup cost)
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            _ = model(inputs)
            break

    total_time = 0
    total_images = 0

    with torch.no_grad():
        for _ in range(num_runs):
            for inputs, _ in test_loader:
                inputs = inputs.to(device)
                batch_size = inputs.size(0)
                start_time = time.time()
                _ = model(inputs)
                end_time = time.time()

                total_time += (end_time - start_time)
                total_images += batch_size

    avg_time_per_image = total_time / total_images
    return avg_time_per_image


In [27]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def calculate_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size = os.path.getsize(filename) / (1024 * 1024)  # Size in MB
    os.remove(filename)
    return size

def compare_model_sizes(teacher, student, pruned_student):
    # Count parameters
    teacher_params = count_parameters(teacher)
    student_params = count_parameters(student)
    pruned_params = count_parameters(pruned_student)
    
    # Calculate disk size
    teacher_size = calculate_model_size(teacher, "teacher.pth")
    student_size = calculate_model_size(student, "student.pth")
    pruned_size = calculate_model_size(pruned_student, "pruned_student.pth")
    
    # Print comparison
    print("\n--- Model Size Comparison ---")
    print(f"Teacher Model: {teacher_params} parameters, {teacher_size:.2f} MB")
    print(f"Student Model (Before Pruning): {student_params} parameters, {student_size:.2f} MB")
    print(f"Student Model (After Pruning): {pruned_params} parameters, {pruned_size:.2f} MB")
    
    # Calculate compression ratio
    compression_ratio = student_size / pruned_size
    print(f"\nCompression Ratio: {compression_ratio:.2f}x")

In [28]:
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001, patience=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    best_val_accuracy = 0.0
    best_model_state = None
    patience_counter = 0  # Counter for early stopping
    
    for epoch in range(epochs):
        print(epoch)
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        # Evaluate on the validation set
        val_accuracy = evaluate(model, val_loader, device)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Accuracy: {val_accuracy:.2f}%")
        
        # Early stopping logic
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
            patience_counter = 0  # Reset patience counter
            torch.save(model.state_dict(), 'best_teacher_model.pth')  # Save the best model
            print(f" New best model saved with validation accuracy: {best_val_accuracy:.2f}%")
        else:
            patience_counter += 1
            print(f" No improvement in validation accuracy ({patience_counter}/{patience})")
            
            # Stop training if no improvement for 'patience' epochs
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
                break
    
    # Load the best model state
    model.load_state_dict(torch.load('best_teacher_model.pth'))
    print("\nLoading the best model for final evaluation.")
    
    # Evaluate on the test set
    test_accuracy = evaluate(model, test_loader, device)
    print(f"Test Accuracy with Best Model: {test_accuracy:.2f}%")
    
    return model



In [29]:
# # Fine-tune the teacher model
# teacher = train_model(teacher, train_loader, val_loader, epochs=200, lr=0.001, patience=5)

In [30]:
def compute_gradient_importance(
    teacher, student, data_loader, device, temperature=4.0, alpha=0.5, beta_prob=0.5, accumulation_epochs=3
):
    importance_scores = {}

    # Initialize importance score storage for conv layer weights only
    for name, param in student.named_parameters():
        if 'weight' in name and len(param.shape) == 4:  # Conv weights only
            importance_scores[name] = torch.zeros_like(param.data, device=device)

    teacher.to(device).eval()
    student.to(device).train()

    # Add momentum for gradient accumulation smoothing
    momentum = 0.9  # Controls exponential moving average
    accumulated_batches = 0  # Track for bias correction

    for epoch in range(accumulation_epochs):
        print(f"Accumulation Epoch {epoch+1}/{accumulation_epochs}")
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            student.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            # Temperature scaling
            student_logits_temp = student_logits / temperature
            teacher_logits_temp = teacher_logits / temperature

            # Compute losses
            distillation_loss = cakld_loss(student_logits_temp, teacher_logits_temp, beta_prob) * (temperature ** 2)
            ce_loss = F.cross_entropy(student_logits, labels)
            loss = alpha * distillation_loss + (1 - alpha) * ce_loss

            # Modified backward propagation
            loss.backward()

            # Accumulate importance scores with parameter-gradient product
            accumulated_batches += 1
            for name, param in student.named_parameters():
                if name in importance_scores and param.grad is not None:
                    # Key modification: Use parameter-gradient product magnitude
                    grad_product = (param.data * param.grad).abs_()
                    
                    # Exponential moving average with bias correction
                    if accumulated_batches == 1:
                        importance_scores[name] = grad_product
                    else:
                        importance_scores[name] = momentum * importance_scores[name] + (1 - momentum) * grad_product

    # Apply bias correction for EMA
    for name in importance_scores:
        importance_scores[name] /= (1 - momentum**accumulated_batches)

    return importance_scores

In [31]:
def gradient_based_global_prune(model, importance_scores, prune_ratio=0.95):
    all_scores = torch.cat([score.flatten() for score in importance_scores.values()])
    threshold = torch.topk(all_scores, k=int(prune_ratio * all_scores.numel()), largest=False)[0][-1]

    for name, param in model.named_parameters():
        if name in importance_scores:
            mask = (importance_scores[name] > threshold).float()
            param.data.mul_(mask)

    return model


In [32]:
import torch
import torch.nn.functional as F
import torch.optim as optim

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

def retrain_with_sparsity(student, train_loader, val_loader, epochs=5, save_path="retrained_student_model.pt", patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9)

    # 1. Store masks AND zero momentum buffers for pruned weights
    masks = {}
    for name, param in student.named_parameters():
        if 'weight' in name and param.dim() == 4:  # Consider only conv layers
            mask = (param != 0).float().to(device)
            masks[name] = mask
            # Zero momentum buffers for pruned weights
            if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                optimizer.state[param]['momentum_buffer'] *= mask

    student = student.to(device)
    best_val_acc = 0.0
    best_model = None
    patience_counter = 0  # Counter for early stopping

    # 2. Add gradient clipping to prevent NaN
    max_grad_norm = 1.0

    for epoch in range(epochs):
        student.train()
        total_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = student(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()

            # Apply masks to gradients
            for name, param in student.named_parameters():
                if name in masks:
                    param.grad.data *= masks[name]

            # Gradient clipping before optimizer step
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_grad_norm)

            optimizer.step()

            # Reapply masks and update momentum buffers
            for name, param in student.named_parameters():
                if name in masks:
                    param.data *= masks[name]
                    if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                        optimizer.state[param]['momentum_buffer'] *= masks[name]

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # Validation phase
        student.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = student(inputs)
                loss = F.cross_entropy(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total

        # Track best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = student.state_dict()
            torch.save(best_model, save_path)
            patience_counter = 0  # Reset patience counter
            print(f"New best model saved with Val Accuracy: {best_val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. No improvement for {patience} epochs.")
                break  # Stop training

        # Print results
        sparsity = calculate_sparsity(student)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_acc:.2f}% | Sparsity: {sparsity*100:.2f}%\n")

    print(f"Best Validation Accuracy: {best_val_acc:.2f}% | Best Model Saved at: {save_path}")
    return student

In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time

# KD training with CA-KLD loss and mask-based momentum handling
def retrain_with_KD(teacher, student, train_loader, val_loader, epochs=50,
                    temperature=5.0, alpha=0.5, beta_prob=0.5, patience=5,
                    save_path="student_before_pruning.pth"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9)

    # 1. Store masks and zero momentum buffers
    masks = {}
    for name, param in student.named_parameters():
        if 'weight' in name and param.dim() == 4:
            mask = (param != 0).float().to(device)
            masks[name] = mask
            if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                optimizer.state[param]['momentum_buffer'] *= mask

    teacher = teacher.to(device).eval()
    student = student.to(device)

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        student.train()
        total_loss, correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            # Apply temperature
            teacher_logits_temp = teacher_logits / temperature
            student_logits_temp = student_logits / temperature

            # Logits normalization
            teacher_logits_temp = normalize(teacher_logits_temp)
            student_logits_temp = normalize(student_logits_temp)


            # CA-KLD loss
            kd_loss = cakld_loss(student_logits_temp, teacher_logits_temp, beta_prob) * (temperature ** 2)
            ce_loss = F.cross_entropy(student_logits, labels)

            loss = alpha * kd_loss + (1 - alpha) * ce_loss
            loss.backward()
            optimizer.step()

            # Reapply masks and update momentum
            for name, param in student.named_parameters():
                if name in masks:
                    param.data *= masks[name]
                    if optimizer.state.get(param, None) and 'momentum_buffer' in optimizer.state[param]:
                        optimizer.state[param]['momentum_buffer'] *= masks[name]

            total_loss += loss.item()
            _, predicted = student_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # Validation
        student.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = student(inputs)
                loss = F.cross_entropy(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total
        sparsity = calculate_sparsity(student) * 100.0  # Assuming this function is defined elsewhere

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Sparsity: {sparsity:.2f}%")

        # Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = student.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. No improvement for {patience} epochs.")
                break

    # Restore and save best model
    student.load_state_dict(best_model_state)
    torch.save(student.state_dict(), save_path)
    print(f"Student model saved before pruning at: {save_path}")
    total_time = time.time() - start_time
    print(f"Total Training Time: {total_time // 60:.0f}m {total_time % 60:.0f}s")

    return student

In [34]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Training function with KD + CA-KLD and logits normalization
def train_kd_pruning(teacher, student, train_loader, val_loader, epochs=50, temperature=5.0, alpha=0.5,
                     beta_prob=0.5, patience=5, save_path="student_before_pruning.pth"):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.SGD(student.parameters(), lr=0.01, momentum=0.9)

    teacher = teacher.to(device)
    student = student.to(device)
    teacher.eval()  # Freeze teacher

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        student.train()
        total_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            # Temperature scaling
            teacher_logits_temp = teacher_logits / temperature
            student_logits_temp = student_logits / temperature

            # Logits normalization
            teacher_logits_temp = normalize(teacher_logits_temp)
            student_logits_temp = normalize(student_logits_temp)

            # CA-KLD loss (normalized logits)
            distillation_loss = cakld_loss(student_logits_temp, teacher_logits_temp, beta_prob) * (temperature ** 2)

            # Cross-entropy loss
            ground_truth_loss = F.cross_entropy(student_logits, labels)

            # Combined loss
            loss = alpha * distillation_loss + (1 - alpha) * ground_truth_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = student_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        # Validation accuracy
        val_acc = evaluate(student, val_loader, device)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = student.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}. No improvement for {patience} epochs.")
                break

    # Load best model state and save
    student.load_state_dict(best_model_state)
    torch.save(student.state_dict(), save_path)
    print(f"Student model saved before pruning at: {save_path}")

    total_time = time.time() - start_time
    print(f"Total Training Time: {total_time // 60:.0f}m {total_time % 60:.0f}s")

    return student

In [35]:
# Load pretrained ResNet-18 (Student Model)
student = models.resnet18(pretrained=True)
# Modify the final fully connected layer for 100 classes (CIFAR-100)
student.fc = nn.Linear(student.fc.in_features, 200)
student = student.to(device)

In [36]:

student = train_kd_pruning(
    teacher, student, train_loader, val_loader,
    epochs=1, temperature=5.0, alpha=0.5,beta_prob=0.5, patience=5,save_path="student_before_pruning.pth"
)


Epoch 1/1 | Train Loss: 33.9820 | Train Acc: 24.51% | Val Acc: 32.07%
Student model saved before pruning at: student_before_pruning.pth
Total Training Time: 1m 29s


In [37]:
# Calculate sparsity
sparsity = calculate_sparsity(student)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")

teacher_accuracy = evaluate(teacher, test_loader, device)
student_accuracy = evaluate(student, test_loader, device)
print(f"Teacher Model Test Accuracy: {teacher_accuracy:.2f}%")
print(f"Student Model Test Accuracy Before Pruning: {student_accuracy:.2f}%")

Sparsity Before Pruning: 0.00%
Teacher Model Test Accuracy: 78.46%
Student Model Test Accuracy Before Pruning: 33.05%


## 93% Sparsity

In [72]:

model_path = 'student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.9918
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 99.14%


<All keys matched successfully>

In [73]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=2,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 17.1451 | Train Acc: 26.04% | Val Loss: 3.0731 | Val Acc: 31.25% | Sparsity: 98.41%
Epoch 2/50 | Train Loss: 15.0543 | Train Acc: 35.87% | Val Loss: 2.8432 | Val Acc: 36.25% | Sparsity: 98.41%
Epoch 3/50 | Train Loss: 14.2694 | Train Acc: 39.73% | Val Loss: 2.6813 | Val Acc: 39.79% | Sparsity: 98.41%
Epoch 4/50 | Train Loss: 13.7415 | Train Acc: 42.27% | Val Loss: 2.6137 | Val Acc: 40.95% | Sparsity: 98.41%
Epoch 5/50 | Train Loss: 13.3248 | Train Acc: 44.11% | Val Loss: 2.5573 | Val Acc: 42.18% | Sparsity: 98.41%
Epoch 6/50 | Train Loss: 13.0038 | Train Acc: 45.61% | Val Loss: 2.5297 | Val Acc: 42.57% | Sparsity: 98.41%
Epoch 7/50 | Train Loss: 12.7381 | Train Acc: 46.86% | Val Loss: 2.4834 | Val Acc: 44.22% | Sparsity: 98.41%
Epoch 8/50 | Train Loss: 12.4779 | Train Acc: 47.95% | Val Loss: 2.4611 | Val Acc: 44.27% | Sparsity: 98.41%
Epoch 9/50 | Train Loss: 12.2603 | Train Acc: 49.04% | Val Loss: 2.5175 | Val Acc: 43.53% | Sparsity: 98.41%
Epoch 10/50 | Train

In [75]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f" Retrained Pruned Student Model Test Accuracy: {student_accuracy:.2f}%")

 Retrained Pruned Student Model Test Accuracy: 47.24%


In [53]:

model_path = 'student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.9443
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 94.39%


<All keys matched successfully>

In [54]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=5,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 12.7236 | Train Acc: 44.61% | Val Loss: 2.5811 | Val Acc: 46.27% | Sparsity: 93.91%
Epoch 2/50 | Train Loss: 10.7833 | Train Acc: 52.84% | Val Loss: 2.2704 | Val Acc: 50.82% | Sparsity: 93.91%
Epoch 3/50 | Train Loss: 9.7294 | Train Acc: 57.51% | Val Loss: 2.1088 | Val Acc: 52.99% | Sparsity: 93.91%
Epoch 4/50 | Train Loss: 8.9390 | Train Acc: 61.30% | Val Loss: 2.1001 | Val Acc: 53.62% | Sparsity: 93.91%
Epoch 5/50 | Train Loss: 8.2700 | Train Acc: 64.34% | Val Loss: 2.1080 | Val Acc: 53.97% | Sparsity: 93.91%
Epoch 6/50 | Train Loss: 7.7380 | Train Acc: 66.90% | Val Loss: 2.0654 | Val Acc: 54.88% | Sparsity: 93.91%
Epoch 7/50 | Train Loss: 7.2563 | Train Acc: 69.14% | Val Loss: 2.0982 | Val Acc: 54.81% | Sparsity: 93.91%
Epoch 8/50 | Train Loss: 6.7990 | Train Acc: 71.55% | Val Loss: 2.0735 | Val Acc: 55.24% | Sparsity: 93.91%
Epoch 9/50 | Train Loss: 6.3778 | Train Acc: 73.43% | Val Loss: 2.0873 | Val Acc: 55.17% | Sparsity: 93.91%
Epoch 10/50 | Train Loss: 

In [55]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f" Retrained Pruned Student Model Test Accuracy: {student_accuracy:.2f}%")

 Retrained Pruned Student Model Test Accuracy: 55.43%


In [58]:

model_path = 'student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.7464
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 74.61%


<All keys matched successfully>

In [59]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=5,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 11.0936 | Train Acc: 50.76% | Val Loss: 2.3329 | Val Acc: 49.53% | Sparsity: 74.35%
Epoch 2/50 | Train Loss: 9.2297 | Train Acc: 58.76% | Val Loss: 2.1137 | Val Acc: 52.82% | Sparsity: 74.35%
Epoch 3/50 | Train Loss: 7.9529 | Train Acc: 64.55% | Val Loss: 1.9849 | Val Acc: 55.34% | Sparsity: 74.35%
Epoch 4/50 | Train Loss: 6.8567 | Train Acc: 69.72% | Val Loss: 1.9369 | Val Acc: 57.08% | Sparsity: 74.35%
Epoch 5/50 | Train Loss: 5.8824 | Train Acc: 74.58% | Val Loss: 1.9402 | Val Acc: 56.95% | Sparsity: 74.35%
Epoch 6/50 | Train Loss: 5.0232 | Train Acc: 78.83% | Val Loss: 1.9714 | Val Acc: 56.71% | Sparsity: 74.35%
Epoch 7/50 | Train Loss: 4.2980 | Train Acc: 82.59% | Val Loss: 2.0038 | Val Acc: 57.18% | Sparsity: 74.35%
Epoch 8/50 | Train Loss: 3.5761 | Train Acc: 86.13% | Val Loss: 2.0416 | Val Acc: 57.08% | Sparsity: 74.35%
Epoch 9/50 | Train Loss: 2.9443 | Train Acc: 89.32% | Val Loss: 2.0094 | Val Acc: 57.07% | Sparsity: 74.35%
Epoch 10/50 | Train Loss: 2

In [60]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f" Retrained Pruned Student Model Test Accuracy: {student_accuracy:.2f}%")

 Retrained Pruned Student Model Test Accuracy: 56.99%


In [69]:

model_path = 'student_before_pruning.pth'
# Load the model weights
student.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
# Calculate sparsity


model=student
import torch
import torch.nn.utils.prune as prune
import torch.nn as nn
import copy
torch.manual_seed(42)
original_init = copy.deepcopy(model.state_dict())  #

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

# 4️ Apply one-shot global magnitude pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,  # L1 magnitude pruning
    amount=0.5019
)

# Optional: Remove pruning reparameterization so `.weight` becomes pruned tensor
for module, param in parameters_to_prune:
    prune.remove(module, 'weight')

# 5️ Get pruning mask
mask_dict = {k: (v != 0).float() for k, v in model.state_dict().items()}

# 6️ Reset remaining weights to their original initialization values
reset_state = {}
for k in original_init.keys():
    if k in mask_dict:
        reset_state[k] = original_init[k] * mask_dict[k]  # keep init where mask=1
    else:
        reset_state[k] = original_init[k]  # non-pruned params like biases


sparsity = calculate_sparsity(model)
print(f"Sparsity Before Pruning: {sparsity * 100:.2f}%")
model.load_state_dict(reset_state)

Sparsity Before Pruning: 50.17%


<All keys matched successfully>

In [70]:

start_time = time.time()
pruned_student = retrain_with_KD(
    teacher, model, train_loader, val_loader,
    epochs=50, temperature=3.0, alpha=0.7, beta_prob=0.5,patience=1,save_path="pruned_student_retrain_KD_90%.pth"
)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"Retraining completed in {elapsed_time / 60:.2f} minutes ({elapsed_time:.2f} seconds)")

Epoch 1/50 | Train Loss: 11.0005 | Train Acc: 50.99% | Val Loss: 2.3116 | Val Acc: 49.79% | Sparsity: 50.02%
Epoch 2/50 | Train Loss: 9.1148 | Train Acc: 59.12% | Val Loss: 2.1114 | Val Acc: 52.59% | Sparsity: 50.02%
Epoch 3/50 | Train Loss: 7.6954 | Train Acc: 65.59% | Val Loss: 2.0839 | Val Acc: 54.40% | Sparsity: 50.02%
Epoch 4/50 | Train Loss: 6.4893 | Train Acc: 71.28% | Val Loss: 1.9930 | Val Acc: 56.12% | Sparsity: 50.02%
Epoch 5/50 | Train Loss: 5.3395 | Train Acc: 76.87% | Val Loss: 2.0150 | Val Acc: 55.60% | Sparsity: 50.02%
Early stopping triggered at epoch 5. No improvement for 1 epochs.
Student model saved before pruning at: pruned_student_retrain_KD_90%.pth
Total Training Time: 7m 51s
Retraining completed in 7.85 minutes (470.82 seconds)


In [71]:
student_accuracy = evaluate(pruned_student, test_loader, device)
print(f" Retrained Pruned Student Model Test Accuracy: {student_accuracy:.2f}%")

 Retrained Pruned Student Model Test Accuracy: 56.91%
